# EcoData · Sesión 12 (U2) · Dimensión CLIMA (Alahin)
**Equipo EcoData** — Lanzeloth Vargas Marichi · Alahin Reyme Ticona Veliz · Big Data, UPeU

Flujo de la dimensión clima: `Open-Meteo API → Kafka (atmos-clima) → Spark Structured Streaming + modelo GBT (MLlib) → Pushgateway → Prometheus → Grafana`

| Sección | Qué demuestra |
|---|---|
| 0 | Preparación |
| 1 | Conexión con la API (clima) |
| 2 | Verificar y contar eventos en Kafka (`atmos-clima`) |
| 3 | Modelo de temperatura (metadatos) |
| 4 | Jobs de streaming (solo estado) |
| 5 | Métricas de clima en Prometheus / Pushgateway |

> Ejecuta las celdas en orden con Shift+Enter. **No** uses "Run All". Requiere Kafka, Pushgateway y Prometheus levantados (Docker). Si el kernel se reinicia, empieza siempre por la sección 0.

## 0. Preparación

In [1]:
import sys, json, time, subprocess, urllib.request, urllib.parse
import pandas as pd
sys.path.insert(0, "/tmp/s12/scripts")
import clima_comun as clima
import openmeteo_clima_a_kafka as pub_clima

BOOTSTRAP = "kafka:9092"
PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS_CANDIDATOS = ["http://prometheus:9090", "http://lambda26-prometheus:9090"]
print("Scripts del proyecto cargados desde /tmp/s12/scripts")

from kafka import KafkaConsumer, TopicPartition
c = KafkaConsumer(bootstrap_servers=BOOTSTRAP)
print("Kafka OK. Topics:", sorted(t for t in c.topics() if t.startswith("atmos")))
c.close()

Scripts del proyecto cargados desde /tmp/s12/scripts
Kafka OK. Topics: ['atmos-aire', 'atmos-clima', 'atmos-negocio']


In [8]:
# MODELADO: métricas guardadas al entrenar (aire y clima)
import json

for nombre, ruta in [("AIRE (PM2.5)",   "/tmp/s12/modelo_pm25/meta.json"),
                     ("CLIMA (temp.)",  "/tmp/s12/modelo_temp/meta.json")]:
    print("=" * 50)
    print(nombre)
    print("=" * 50)
    try:
        meta = json.load(open(ruta))
        for k, v in meta.items():
            if isinstance(v, dict):
                print(f"{k}:")
                for k2, v2 in v.items():
                    print(f"   {k2:<22}{v2}")
            else:
                print(f"{k:<25}{v}")
    except Exception as e:
        print("no se pudo leer:", e)

AIRE (PM2.5)
persistencia_mae         0.6871207784773903
lineal_mae               0.7419191622745201
gbt_mae                  0.4733888743356669
mejora_vs_persistencia_pct31.105434566443726
umbral_relativo_p95      10.5
n_train                  6988
n_test                   1747
features                 ['l0', 'd1', 'd3', 'd24', 'm24', 'pm10d', 'hs', 'hc']
test_desde               1697774400000
unidad                   ug/m3
fuente                   Open-Meteo Air Quality (CAMS, modelado, ~45 km)
CLIMA (temp.)
persistencia_mae         1.096336576989124
persistencia_diaria_mae  0.6981682884945623
lineal_mae               0.6319017772184087
gbt_mae                  0.5368773383112271
mejora_vs_persistencia_pct51.02987991282233
mejora_vs_diaria_pct     23.102016067089156
n_train                  6988
n_test                   1747
features                 ['l0', 'd1', 'd3', 'd24', 'm24', 'hum', 'humd', 'viento', 'hs', 'hc']
test_desde               1697774400000
unidad                   gr

## 1. Conexión con la API (Open-Meteo, clima)
Open-Meteo entrega datos **horarios y modelados** (reanálisis/pronóstico), no sensores en vivo. La función `desde_api()` del productor de clima consulta la API y devuelve las horas ya ocurridas.

In [2]:
serie_clima = pub_clima.desde_api()
df_clima = pd.DataFrame(serie_clima)
print(f"{len(df_clima)} horas recibidas de la API")
df_clima.tail(8)

69 horas recibidas de la API


,0,1,2,3
61,1793970000000,12.4,47,5.1
62,1793973600000,14.7,39,2.7
63,1793977200000,16.7,32,1.7
64,1793980800000,18.5,26,5.5
65,1793984400000,19.6,23,10.7
66,1793988000000,19.0,22,13.5
67,1793991600000,19.1,21,11.4
68,1793995200000,18.2,27,17.8


## 2. Verificar y contar eventos en Kafka (`atmos-clima`)
El campo `origen` dice de dónde vino cada dato: `openmeteo-api` (API), `openmeteo-replay` (histórico reproducido) o `demo-tardio` (dato tardío de prueba).

In [3]:
c = KafkaConsumer("atmos-clima", bootstrap_servers=BOOTSTRAP, auto_offset_reset="earliest",
                  enable_auto_commit=False, consumer_timeout_ms=8000)
todos = [json.loads(m.value.decode("utf-8")) for m in c]
c.close()
df_k = pd.DataFrame(todos)
print(len(df_k), "eventos leídos en atmos-clima")
if "origen" in df_k:
    display(df_k["origen"].value_counts())
if "timestamp" in df_k:
    print("Horas distintas:", df_k["timestamp"].nunique())
    df_k["hora_UTC"] = pd.to_datetime(df_k["timestamp"], unit="ms")
df_k.tail(10)

7435 eventos leídos en atmos-clima


origen
openmeteo-replay    7234
openmeteo-api        200
demo-tardio            1
Name: count, dtype: int64

Horas distintas: 6987


,tipoEvento,sensorId,temp,hum,viento,origen,timestamp,hora_UTC
7425,clima.lectura,openmeteo-clima-juliaca,14.0,62.0,1.1,openmeteo-replay,1702483200000,2023-12-13 16:00:00
7426,clima.lectura,openmeteo-clima-juliaca,14.9,61.0,2.3,openmeteo-replay,1702486800000,2023-12-13 17:00:00
7427,clima.lectura,openmeteo-clima-juliaca,15.4,57.0,2.1,openmeteo-replay,1702490400000,2023-12-13 18:00:00
7428,clima.lectura,openmeteo-clima-juliaca,14.6,62.0,14.0,openmeteo-replay,1702494000000,2023-12-13 19:00:00
7429,clima.lectura,openmeteo-clima-juliaca,14.4,59.0,12.4,openmeteo-replay,1702497600000,2023-12-13 20:00:00
7430,clima.lectura,openmeteo-clima-juliaca,12.1,67.0,17.6,openmeteo-replay,1702501200000,2023-12-13 21:00:00
7431,clima.lectura,openmeteo-clima-juliaca,10.7,76.0,4.3,openmeteo-replay,1702504800000,2023-12-13 22:00:00
7432,clima.lectura,openmeteo-clima-juliaca,9.5,80.0,2.9,openmeteo-replay,1702508400000,2023-12-13 23:00:00
7433,clima.lectura,openmeteo-clima-juliaca,9.1,87.0,12.3,openmeteo-replay,1702512000000,2023-12-14 00:00:00
7434,clima.lectura,openmeteo-clima-juliaca,9.3,86.0,6.9,openmeteo-replay,1702515600000,2023-12-14 01:00:00


## 3. Modelo de temperatura
GBT de MLlib que predice el **cambio** de temperatura a +1 h; se compara contra la persistencia y la persistencia diaria.

In [4]:
import os
ruta = "/tmp/s12/modelo_temp/meta.json"
if os.path.exists(ruta):
    meta = json.load(open(ruta))
    print("Meta del modelo de clima:")
    for k, v in meta.items():
        print(f"  {k}: {v}")
else:
    print("No se encontró", ruta, "- revisa que el modelo esté en /tmp/s12/modelo_temp")

Meta del modelo de clima:
  persistencia_mae: 1.096336576989124
  persistencia_diaria_mae: 0.6981682884945623
  lineal_mae: 0.6319017772184087
  gbt_mae: 0.5368773383112271
  mejora_vs_persistencia_pct: 51.02987991282233
  mejora_vs_diaria_pct: 23.102016067089156
  n_train: 6988
  n_test: 1747
  features: ['l0', 'd1', 'd3', 'd24', 'm24', 'hum', 'humd', 'viento', 'hs', 'hc']
  test_desde: 1697774400000
  unidad: grados C
  fuente: Open-Meteo (temperature_2m, relative_humidity_2m, wind_speed_10m)


## 4. Jobs de streaming (solo estado)
Spark lee de Kafka, calcula ventanas de 6 h con watermark de 2 h, infiere y empuja métricas al Pushgateway. **Un solo job por tipo**: no lances otro si ya están corriendo.

In [5]:
def jobs_activos():
    out = subprocess.run("ps aux | grep s12_streaming | grep -v grep", shell=True, capture_output=True, text=True).stdout
    return [l for l in out.splitlines() if "spark-submit" in l or "SparkSubmit" in l]
print("Procesos spark activos:", len(jobs_activos()))
for l in jobs_activos():
    print(" ", l[:140])

Procesos spark activos: 0


## 5. Métricas de clima en Prometheus (lo mismo que ve Grafana)

In [6]:
def prom(q):
    for base in PROMETHEUS_CANDIDATOS:
        try:
            u = f"{base}/api/v1/query?" + urllib.parse.urlencode({"query": q})
            return json.load(urllib.request.urlopen(u, timeout=4))["data"]["result"]
        except Exception as e:
            err = e
    raise err

consultas = [
    ("Frescura clima (s)", 'time()-push_time_seconds{job="lambda26_clima",instance="inferencia"}'),
    ("Temperatura última (°C)", "lambda26_clima_temp_ultima"),
    ("Predicción próxima hora (°C)", "lambda26_clima_pred_temp_proxima_hora"),
    ("MAE modelo (°C)", "lambda26_clima_mae_modelo"),
    ("MAE persistencia (°C)", "lambda26_clima_mae_persistencia"),
    ("Mejora vs persistencia (%)", "lambda26_clima_mejora_pct"),
    ("Horas evaluadas", "lambda26_clima_n_evaluadas"),
    ("Heladas detectadas", "lambda26_clima_heladas_total"),
    ("Filas descartadas (watermark)", "lambda26_clima_filas_descartadas_watermark_total"),
]
for nombre, q in consultas:
    try:
        r = prom(q)
        print(f"{nombre:34s}", [round(float(x["value"][1]), 3) for x in r] or "sin datos")
    except Exception as e:
        print(nombre, "->", e)

Frescura clima (s)                 [6587.818]
Temperatura última (°C)            [19.5]
Predicción próxima hora (°C)       [20.212]
MAE modelo (°C)                    [0.451]
MAE persistencia (°C)              [1.281]
Mejora vs persistencia (%)         [64.788]
Horas evaluadas                    [6936.0]
Heladas detectadas                 [295.0]
Filas descartadas (watermark)      [0.0]


Alternativa directa: leer el Pushgateway.

In [7]:
txt = urllib.request.urlopen(PUSHGATEWAY + "/metrics", timeout=4).read().decode()
print("\n".join(l for l in txt.splitlines() if l.startswith("lambda26_clima_") and not l.startswith("#")))

lambda26_clima_estado_filas{instance="streaming",job="lambda26_clima"} 2
lambda26_clima_eventos_total{instance="inferencia",job="lambda26_clima"} 6986
lambda26_clima_filas_descartadas_watermark_total{instance="streaming",job="lambda26_clima"} 0
lambda26_clima_heladas_total{instance="inferencia",job="lambda26_clima"} 295
lambda26_clima_hum_ultima{instance="inferencia",job="lambda26_clima"} 21
lambda26_clima_input_rows_per_second{instance="streaming",job="lambda26_clima"} 0
lambda26_clima_lote_id{instance="inferencia",job="lambda26_clima"} 0
lambda26_clima_lote_ventanas{instance="streaming",job="lambda26_clima"} 1
lambda26_clima_mae_modelo{instance="inferencia",job="lambda26_clima"} 0.4511594997465069
lambda26_clima_mae_persistencia{instance="inferencia",job="lambda26_clima"} 1.2812716262975765
lambda26_clima_mejora_pct{instance="inferencia",job="lambda26_clima"} 64.78814558235408
lambda26_clima_n_evaluadas{instance="inferencia",job="lambda26_clima"} 6936
lambda26_clima_pred_temp_proxima

**Dashboard Grafana:** http://localhost:43000/d/lambda26-s12-ecodata (admin / admin) · **Prometheus:** http://localhost:49090 · **Kafka UI:** http://localhost:48085